In [1]:
import json
import random
from collections import defaultdict
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split


In [2]:
data_path = "../dataset/main_1mv_train/all_text_serialized_questions.json"

with open(data_path, "r") as f:
    data = json.load(f)

In [3]:
# Process the data to create a new list of samples with fields "question" and "answer"
# The question field is created as: "\n".join(str(frame) for frame in row["sequence_json"]) + "\n" + row["question"]
formatted_data = []
for row in data:
    frames_text = "\n".join(str(frame) for frame in row["sequence_json"])
    full_question = frames_text + "\n" + row["question"]
    
    # Build the new sample (only keep question and answer)
    sample = {"question": full_question} | {k: str(row[k]) if k == 'answer' else row[k] for k in ['seq_len', 'answer', 'qid', 'seq_len', 'qtype', 'atype']}
    formatted_data.append(sample)
    # Group by seq_len
    seq_len = row["seq_len"]


In [4]:
# Step 1: Group by seq_len
grouped_by_seq_len = defaultdict(list)
for sample in formatted_data:
    grouped_by_seq_len[sample["seq_len"]].append(sample)

# Step 2: Within each seq_len group, perform stratified train-test split by qtype
final_datasets = {}

for seq_len, samples in grouped_by_seq_len.items():
    # Organize by qtype for stratified splitting
    qtypes = [s["qtype"] for s in samples]

    # If there's only one unique qtype, we perform a random split (since stratification requires >1 class)
    if len(set(qtypes)) == 1:
        train_samples, test_samples = train_test_split(samples, test_size=0.2, random_state=42)
    else:
        train_samples, test_samples = train_test_split(samples, test_size=0.2, random_state=42, stratify=qtypes)

    # Convert to Hugging Face datasets
    ds_train = Dataset.from_list(train_samples, split="train")
    ds_test = Dataset.from_list(test_samples, split="val")

    # Store in a DatasetDict under the seq_len key
    final_datasets[f"seq_len_{seq_len}"] = DatasetDict({
        "train": ds_train,
        "val": ds_test
    })


In [5]:
from pprint import pprint
from datasets import load_dataset

In [6]:
pprint(final_datasets['seq_len_1']['val'][:5])

{'answer': ['Bathroom', '1', '0', 'Bedroom', 'Nobody'],
 'atype': ['room', 'number', 'number', 'room', 'person'],
 'qid': ['0004359', '0003328', '0005468', '0001191', '0001745'],
 'qtype': ['crowded_room',
           'n_char_at_frame',
           'steps_in_room',
           'char_at_frame',
           'last_at_room'],
 'question': ["{'step_id': 1, 'rooms': {'Kitchen': ['Mary'], 'Bathroom': "
              "['Sandra', 'Daniel', 'Michael'], 'Garden': ['John'], 'Office': "
              "[], 'Bedroom': [], 'Hallway': []}}\n"
              'Which room was crowded (3 or more people in one room) for the '
              'most steps?',
              "{'step_id': 1, 'rooms': {'Kitchen': [], 'Bathroom': ['Sandra', "
              "'Michael'], 'Garden': ['John'], 'Office': [], 'Bedroom': "
              "['Daniel'], 'Hallway': ['Mary']}}\n"
              'How many other characters were in the same room as Sandra at '
              'step 1?',
              "{'step_id': 1, 'rooms': {'Kitchen': [], 

In [7]:
test_data_path = "/workspace-SR004.nfs2/data/long_vqa_synth/main_1mv/all_text_serialized_questions.json"

with open(test_data_path, "r") as f:
    test_data = json.load(f)

In [8]:
# Process the data to create a new list of samples with fields "question" and "answer"
# The question field is created as: "\n".join(str(frame) for frame in row["sequence_json"]) + "\n" + row["question"]
formatted_test_data = []
for row in test_data:
    frames_text = "\n".join(str(frame) for frame in row["sequence_json"])
    full_question = frames_text + "\n" + row["question"]
    
    # Build the new sample (only keep question and answer)
    sample = {"question": full_question} | {k: str(row[k]) if k == 'answer' else row[k] for k in ['seq_len', 'answer', 'qid', 'seq_len', 'qtype', 'atype']}
    formatted_test_data.append(sample)
    # Group by seq_len
    seq_len = row["seq_len"]


In [12]:
test_grouped_by_seq_len = defaultdict(list)

for sample in formatted_test_data:
    test_grouped_by_seq_len[sample["seq_len"]].append(sample)

for seq_len, samples in test_grouped_by_seq_len.items():
    ds = Dataset.from_list(samples, split="test")

    # Store in a DatasetDict under the seq_len key
    if f"seq_len_{seq_len}" in final_datasets:
        final_datasets[f"seq_len_{seq_len}"]["test"] = ds
    else:
        final_datasets[f"seq_len_{seq_len}"] = DatasetDict({"test": ds})


In [13]:
final_datasets.keys()

dict_keys(['seq_len_1', 'seq_len_2', 'seq_len_4', 'seq_len_8', 'seq_len_16', 'seq_len_32', 'seq_len_64', 'seq_len_128'])

In [14]:
hf_dataset_path = "../dataset/hf_main_1mv_train"
for seq_len, dataset in final_datasets.items():
    dataset.save_to_disk(f"{hf_dataset_path}/{seq_len}")

Saving the dataset (0/1 shards):   0%|          | 0/4800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

In [26]:
import yaml

# Simulating the existing dataset structure
seq_len_groups = list(final_datasets.keys())  # Replace with actual seq_len groups

# Construct the YAML structure
yaml_data = {
    "configs": [
        {
            "config_name": "default",
            "data_files": [
                {"split": "train", "path": "*/train/*.arrow"},
                {"split": "val", "path": "*/val/*.arrow"},
                {"split": "test", "path": "*/test/*.arrow"},
            ]
        }
    ]
}

# Add specific configurations for each seq_len group
for seq_len in seq_len_groups:
    yaml_data["configs"].append({
        "config_name": seq_len,
        "data_files": [
            {"split": "train", "path": f"{seq_len}/train/*.arrow"} if "train" in final_datasets[seq_len].keys() else None,
            {"split": "val", "path": f"{seq_len}/val/*.arrow"} if "val" in final_datasets[seq_len].keys() else None,
            {"split": "test", "path": f"{seq_len}/test/*.arrow"},
        ]
    })

# Write YAML to file
yaml_file_path = f"{hf_dataset_path}/README.md"
with open(yaml_file_path, "w") as yaml_file:
    yaml.dump(yaml_data, yaml_file, default_flow_style=False, explicit_start=True, explicit_end=True, sort_keys=False)

print(f"YAML configuration saved to {yaml_file_path}")


YAML configuration saved to ../dataset/hf_main_1mv_train/README.md


In [27]:
len_ds = load_dataset(hf_dataset_path, "seq_len_64")

Generating test split: 0 examples [00:00, ? examples/s]